# Index/Debt/Equity Signal Demonstration

This notebook demonstrates **19 signals** for index/debt/equity strategies:
- **11 Index-Level Signals**: Market-level factors derived from indices (CDX, HYG, VIX, ES)
- **8 Single-Name Signals**: Issuer/security-level factors

Pattern follows the debt-equity strategy with Polars-based processing.

In [1]:
import polars as pl
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, Tuple
import re

## Configuration

In [2]:
@dataclass
class SignalConfig:
    """Configuration for signal computation."""
    half_life_hl_mean: int = 60
    half_life_hl_std: int = 252
    clip_bounds: Tuple[float, float] = (-3.0, 3.0)
    lookback_days: int = 20
    vol_lookback: int = 60
    correlation_window: int = 60
    
    # Asset pair routing
    main_pairs: list = field(default_factory=lambda: ['hyg|spy', 'jnk|qqq', 'lqd|iwm'])
    cdxig_pairs: list = field(default_factory=lambda: ['cdxig|spx', 'cdxhy|ndx'])
    
    # Sign flip patterns
    sign_flip_patterns: list = field(default_factory=lambda: ['bkln|rtyi', 'lqd_rh|esi'])


def get_ewm_alpha(half_life: int) -> float:
    """Convert half-life to EWM alpha."""
    return 1 - np.exp(-np.log(2) / half_life)


def should_flip_sign(pair_name: str, config: SignalConfig) -> bool:
    """Check if signal should be flipped for this pair."""
    for pattern in config.sign_flip_patterns:
        if re.search(pattern, pair_name, re.IGNORECASE):
            return True
    return False


def get_data_source(pair_name: str, config: SignalConfig) -> str:
    """Determine data source for pair."""
    for pattern in config.cdxig_pairs:
        if re.search(pattern.split('|')[0], pair_name, re.IGNORECASE):
            return 'cdxig'
    return 'main'

## Signal Utilities

Import production utilities from [signal_utils.py](signal_utils.py) (located in notebooks folder).

In [ ]:
# Import signal utilities from local file
from signal_utils import (
    apply_scaling,
    apply_filter,
    get_filter,
    compute_inv_vol_weights,
    apply_signal_to_asset,
    compute_signal_returns,
    zscore_ewm,
    compute_rolling_zscore,
    compute_kalman_beta,
    KalmanFilter1D,
    apply_kalman_filter,
    kalman_smooth_spread,
    compute_momentum,
    compute_volatility,
    compute_correlation,
    sign_flip_for_pair,
    route_to_index,
)

# Notebook-specific helpers
def get_ewm_alpha(half_life: int) -> float:
    """Convert half-life to EWM alpha parameter."""
    return 2 / (half_life + 1)


def compute_ewm_zscore(
    df: pl.DataFrame,
    value_col: str,
    config: SignalConfig,
    output_col: str = 'signal'
) -> pl.DataFrame:
    """Compute EWM z-score normalization."""
    alpha_mean = get_ewm_alpha(config.half_life_hl_mean)
    alpha_std = get_ewm_alpha(config.half_life_hl_std)
    
    return df.with_columns([
        pl.col(value_col).ewm_mean(alpha=alpha_mean).alias('_ewm_mean'),
        pl.col(value_col).ewm_std(alpha=alpha_std).alias('_ewm_std'),
    ]).with_columns([
        ((pl.col(value_col) - pl.col('_ewm_mean')) / pl.col('_ewm_std'))
        .clip(config.clip_bounds[0], config.clip_bounds[1])
        .alias(output_col)
    ]).drop(['_ewm_mean', '_ewm_std'])


def compute_momentum_signal(
    df: pl.DataFrame,
    price_col: str,
    lookback: int,
    config: SignalConfig,
    output_col: str = 'signal'
) -> pl.DataFrame:
    """Compute momentum with EWM normalization."""
    return df.with_columns([
        (pl.col(price_col) / pl.col(price_col).shift(lookback) - 1).alias('_return')
    ]).pipe(compute_ewm_zscore, '_return', config, output_col).drop('_return')


def compute_volatility_signal(
    df: pl.DataFrame,
    return_col: str,
    window: int,
    config: SignalConfig,
    output_col: str = 'signal'
) -> pl.DataFrame:
    """Compute rolling volatility with EWM normalization."""
    return df.with_columns([
        pl.col(return_col).rolling_std(window).alias('_vol')
    ]).pipe(compute_ewm_zscore, '_vol', config, output_col).drop('_vol')


print("Signal utilities imported from signal_utils.py")

---
# INDEX-LEVEL SIGNALS (11)
Market-level signals derived from index data sources (CDX, HYG, VIX, ES futures).
---

## S1: Spread Reversion (Macro)

In [ ]:
"""
Macro Signal S1: Spread Reversion.

Signal Logic: EWM smooth spread, take diff, normalize by EWM std of smoothed spread.
Per-pair routing: main pairs use 'main' source, others use 'cdxig'.
Sign flip: bkln|rty1 and lqd_rh|esi get negated score.
"""

def compute_score_s1(
    spread_df: pl.DataFrame, 
    asset: str, 
    hl_mean: int, 
    hl_std: int
) -> pl.DataFrame:
    """Spread reversion score: diff(ewm_mean(spread)) / ewm_std(ewm_mean(spread))."""
    return (
        spread_df.filter(pl.col("asset") == asset)
        .select("date", "spread")
        .sort("date")
        .with_columns(pl.col("spread").fill_nan(None).forward_fill())
        .with_columns(ewm=pl.col("spread").ewm_mean(half_life=hl_mean))
        .with_columns(
            score=pl.col("ewm").diff() / pl.col("ewm").ewm_std(half_life=hl_std)
        )
        .with_columns(score=pl.col("score").fill_nan(None).forward_fill())
        .select("date", "score")
    )


def apply_scaling_s1(df: pl.DataFrame, bounds: Tuple[float, float]) -> pl.DataFrame:
    """Clip scores to bounds."""
    return df.with_columns(
        pl.col("score").clip(bounds[0], bounds[1])
    )


def compute_signal_s1_spread_reversion(config: dict) -> pl.DataFrame:
    """S1 Macro: Spread Reversion
    
    Full production-style implementation.
    """
    sig_cfg = config["macro_signals"]["spread_reversion"]
    clip_bounds = tuple(config["clip_bounds"])
    
    # In production: jpm_df = read_data(config)
    # Here we use mock data or loaded DataFrame
    jpm_df = config.get("data_df", pl.DataFrame())
    
    # Compute, zscore, filter per source index
    scores = {}
    for index in ["cdxig", "main"]:
        df = compute_score_s1(
            jpm_df, index, hl_mean=sig_cfg["hl_mean"], hl_std=sig_cfg["hl_std"]
        )
        df = apply_scaling_s1(df, bounds=clip_bounds)
        # In production: df = apply_filter(df, de_filter)
        scores[index] = df
    
    # Build per-pair score DataFrame [date, asset_pair, score]
    asset_pairs = config["data"]["returns_streams"]["de_assets"]
    pair_scores = []
    for ap in asset_pairs:
        # Route to correct source index
        df = scores["main"] if ap in ("main|vg1", "xover|vg1") else scores["cdxig"]
        # Sign flip for specific pairs
        mult = -1.0 if ap in ("bkln|rty1", "lqd_rh|esi") else 1.0
        pair_scores.append(
            df.with_columns(
                score=pl.col("score") * mult,
                asset_pair=pl.lit(ap).cast(pl.Categorical),
            )
        )
    
    score_df: pl.DataFrame = pl.concat(pair_scores).select(
        "date", "asset_pair", "score"
    )
    
    # In production: signal_df = apply_signal_to_asset(score_df, config)
    signal_df = score_df.with_columns(
        signal_name=pl.lit("spread_reversion").cast(pl.Categorical)
    )
    
    return signal_df


print("S1 Macro: Spread Reversion")
print("  - compute_score_s1(): EWM smooth -> diff -> normalize by EWM std")
print("  - Per-index routing: main vs cdxig")
print("  - Sign flip: bkln|rty1, lqd_rh|esi -> negated")

S1 Macro: Spread Reversion - Mean-reversion on credit spreads


## S2: Equity Momentum (Macro)

In [ ]:
"""
Macro Signal S2: Equity Momentum.

Score = EWM mean of ES1 pct_change, z-scored by EWM std.
Data source: Bloomberg futures data (ES1 - S&P 500 front month future).
"""

def read_data_s2(config: dict) -> pl.DataFrame:
    """Load futures data (cached with full config tickers).
    
    In production: return read_futures_data(
        start_date=date.fromisoformat(config["start_date"]),
        end_date=date.fromisoformat(config["end_date"]),
        table=cfg["table"],
        tickers=cfg["tickers"],
    )
    """
    cfg = config["data"]["jade_tables"]["bbg_futures"]
    # Mock: return loaded data or empty DataFrame
    return config.get("futures_df", pl.DataFrame())


def compute_score_s2(fut_df: pl.DataFrame, lookback: int) -> pl.DataFrame:
    """Equity momentum raw score: EWM mean of ES1 daily returns."""
    return (
        fut_df.filter(pl.col("asset") == "es1")
        .select("date", "close")
        .sort("date")
        .with_columns(pl.col("close").pct_change().alias("ret"))
        .with_columns(
            score=pl.col("ret").ewm_mean(half_life=lookback)
        )
        .with_columns(score=pl.col("score").fill_nan(None).forward_fill())
        .select("date", "score")
    )


def apply_scaling_s2(
    df: pl.DataFrame, 
    ewm_std_hl: int, 
    bounds: Tuple[float, float]
) -> pl.DataFrame:
    """Z-score by EWM std, then clip to bounds."""
    return df.with_columns(
        score=pl.col("score") / pl.col("score").ewm_std(half_life=ewm_std_hl)
    ).with_columns(
        score=pl.col("score").clip(bounds[0], bounds[1])
    )


def compute_signal_s2_equity_momentum(config: dict) -> pl.DataFrame:
    """S2 Macro: Equity Momentum
    
    Full production-style implementation.
    """
    sig_cfg = config["macro_signals"]["equity_momentum"]
    clip_bounds = tuple(config["clip_bounds"])
    
    fut_df = read_data_s2(config)
    df = compute_score_s2(fut_df, lookback=sig_cfg["hl_mean"])
    df = apply_scaling_s2(df, ewm_std_hl=sig_cfg["hl_std"], bounds=clip_bounds)
    
    # Get filter configuration
    filter_cfg = config["filters"]
    # In production: de_filter = get_filter(
    #     date.fromisoformat(config["start_date"]),
    #     date.fromisoformat(config["end_date"]),
    #     config["data"],
    #     level=filter_cfg["level"],
    #     window=filter_cfg["window"],
    #     asset=filter_cfg["asset"],
    # )
    
    # In production: df = apply_filter(df, de_filter)
    # In production: signal_df = apply_signal_to_asset(df, config)
    signal_df = df.with_columns(
        signal_name=pl.lit("equity_momentum").cast(pl.Categorical)
    )
    
    return signal_df


print("S2 Macro: Equity Momentum")
print("  - read_data_s2(): Load Bloomberg futures (ES1)")
print("  - compute_score_s2(): EWM mean of daily returns")
print("  - apply_scaling_s2(): Z-score by EWM std -> clip to bounds")

S2 Macro: Equity Momentum - SPX return momentum


## S3: Cash Dispersion (Macro)

In [ ]:
"""
Macro Signal S3: Cash Dispersion Ratio.

Signal Logic: Rolling median → EWM smooth → quantile clip → sign on daily data.
Already internally normalized – no external zscore.
Data source: JPM DQ (indices=["dispersion"], fields=["cash_dispersion_ratio"]).
"""

def read_data_s3(config: dict) -> pl.DataFrame:
    """Load cash dispersion ratio from JPM DQ (separate cache entry from standard JPM read).
    
    In production: return read_jpm_dq_data(
        start_date=date.fromisoformat(config["start_date"]),
        end_date=date.fromisoformat(config["end_date"]),
        table=cfg["table"],
        indices=["dispersion"],
        fields=["cash_dispersion_ratio"],
    )
    """
    cfg = config["data"]["jade_tables"]["jpm_dq"]
    return config.get("cdr_df", pl.DataFrame())


def compute_score_s3(cdr_df: pl.DataFrame, sig_cfg: dict) -> pl.DataFrame:
    """CDR signal: rolling median → EWM smooth → quantile clip → sign on daily data.
    
    Returns DataFrame[date, score] at daily frequency.
    """
    df = cdr_df.sort("date")
    df = df.with_columns(pl.col("cash_dispersion_ratio").forward_fill().alias("cdr"))
    df = df.filter(pl.col("cdr").is_not_null())
    
    # Rolling median → EWM smooth on daily data
    df = df.with_columns(
        pl.col("cdr")
        .rolling_median(window_size=sig_cfg["hl_mean"])
        .ewm_mean(half_life=sig_cfg["hl_mean"])
        .alias("cdr_slow")
    )
    
    # Quantile clip on daily data
    df = df.with_columns([
        pl.col("cdr_slow")
        .rolling_quantile(
            sig_cfg["q_lower"],
            window_size=sig_cfg["q_window"],
            interpolation="linear",
        )
        .alias("q_lower"),
        pl.col("cdr_slow")
        .rolling_quantile(
            sig_cfg["q_upper"],
            window_size=sig_cfg["q_window"],
            interpolation="linear",
        )
        .alias("q_upper"),
    ])
    
    # Clip to quantile bounds
    df = df.with_columns(
        pl.when(
            (pl.col("cdr_slow") < pl.col("q_lower"))
            | (pl.col("cdr_slow") > pl.col("q_upper"))
        )
        .then(0.0)
        .otherwise(pl.col("cdr_slow"))
        .alias("cdr_clipped")
    )
    
    # Sign of clipped value as score
    df = df.with_columns(pl.col("cdr_clipped").sign().alias("score"))
    
    return df.select("date", "score")


def compute_signal_s3_cash_dispersion(config: dict) -> pl.DataFrame:
    """S3 Macro: Cash Dispersion
    
    Full production-style implementation.
    No zscore - signal is already internally normalized to {-1, 0, 1}.
    """
    sig_cfg = config["macro_signals"]["cash_dispersion"]
    cdr_df = read_data_s3(config)
    df = compute_score_s3(cdr_df, sig_cfg)
    
    # No zscore - signal is already internally normalized to {-1, 0, 1}
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("cash_dispersion").cast(pl.Categorical)
    )
    
    return signal_df


print("S3 Macro: Cash Dispersion Ratio")
print("  - read_data_s3(): Load from JPM DQ (dispersion indices)")
print("  - compute_score_s3():")
print("    1. Rolling median → EWM smooth")
print("    2. Rolling quantile bounds (q_lower, q_upper)")
print("    3. Clip outside bounds → sign() as final score")
print("  - Output: {-1, 0, 1} - no external z-score needed")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S3 Macro: Cash Dispersion - Cross-sectional return dispersion


## S4: Credit Momentum (Macro)

In [ ]:
"""
Macro Signal S4: CDX HY Momentum.

Score = EWM mean of CDX HY total return level, z-scored by EWM std.
Data source: JPM DQ data (indices config).
"""

def read_data_s4(config: dict) -> pl.DataFrame:
    """Load JPM DQ data (cached with full config indices).
    
    In production: return read_jpm_dq_data(
        start_date=date.fromisoformat(config["start_date"]),
        end_date=date.fromisoformat(config["end_date"]),
        table=cfg["table"],
        indices=cfg["indices"],
    )
    """
    cfg = config["data"]["jade_tables"]["jpm_dq"]
    return config.get("jpm_df", pl.DataFrame())


def compute_score_s4(jpm_df: pl.DataFrame, hl_mean: int) -> pl.DataFrame:
    """CDX HY momentum raw score: EWM mean of raw total return level."""
    return (
        jpm_df.filter(pl.col("asset") == "cdxhy")
        .select("date", "total_return")
        .sort("date")
        .with_columns(
            score=pl.col("total_return").ewm_mean(half_life=hl_mean)
        )
        .drop_nulls("score")
        .select("date", "score")
    )


def compute_signal_s4_credit_momentum(config: dict) -> pl.DataFrame:
    """S4 Macro: CDX HY Momentum
    
    Full production-style implementation.
    """
    sig_cfg = config["macro_signals"]["credit_momentum"]
    clip_bounds = tuple(config["clip_bounds"])
    
    jpm_df = read_data_s4(config)
    df = compute_score_s4(jpm_df, hl_mean=sig_cfg["hl_mean"])
    df = apply_scaling(df, ewm_std_hl=sig_cfg["hl_std"], bounds=clip_bounds)
    
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("credit_momentum").cast(pl.Categorical)
    )
    
    return signal_df


print("S4 Macro: CDX HY Momentum")
print("  - read_data_s4(): Load JPM DQ data (cdxhy index)")
print("  - compute_score_s4(): total_return -> ewm_mean")
print("  - apply_scaling(): z-score by EWM std -> clip to bounds")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S4 Macro: Credit Momentum - CDX index momentum


## S5: Credit Volatility (Macro)

In [ ]:
"""
Macro Signal S5: VTRX Momentum (Credit Volatility).

Score = EWM mean of VTRX pct_change, z-scored by EWM std.
Data source: Bloomberg CDX vol data (vtrxuhp - VTRX index).
"""

def read_data_s5(config: dict) -> pl.DataFrame:
    """Load VTRX data (cached with full config tickers).
    
    In production: return read_vtrx_data(
        start_date=date.fromisoformat(config["start_date"]),
        end_date=date.fromisoformat(config["end_date"]),
        table=cfg["table"],
        tickers=cfg["tickers"],
    )
    """
    cfg = config["data"]["jade_tables"]["bbg_cdx_vol"]
    return config.get("vtrx_df", pl.DataFrame())


def compute_score_s5(vtrx_df: pl.DataFrame, hl_mean: int) -> pl.DataFrame:
    """VTRX momentum raw score: EWM mean of pct_change."""
    return (
        vtrx_df.filter(pl.col("asset") == "vtrxuhp")
        .sort("date")
        .with_columns(
            score=pl.col("close")
            .forward_fill()
            .pct_change()
            .ewm_mean(half_life=hl_mean)
        )
        .drop_nulls("score")
        .select("date", "score")
    )


def compute_signal_s5_credit_vol(config: dict) -> pl.DataFrame:
    """S5 Macro: VTRX Momentum (Credit Volatility)
    
    Full production-style implementation.
    """
    sig_cfg = config["macro_signals"]["credit_vol"]
    clip_bounds = tuple(config["clip_bounds"])
    
    vtrx_df = read_data_s5(config)
    df = compute_score_s5(vtrx_df, hl_mean=sig_cfg["hl_mean"])
    df = apply_scaling(df, ewm_std_hl=sig_cfg["hl_std"], bounds=clip_bounds)
    
    filter_cfg = config["filters"]
    # In production: de_filter = get_filter(
    #     date.fromisoformat(config["start_date"]),
    #     date.fromisoformat(config["end_date"]),
    #     config["data"],
    #     level=filter_cfg["level"],
    #     window=filter_cfg["window"],
    #     asset=filter_cfg["asset"],
    # )
    
    # In production: df = apply_filter(df, de_filter)
    # In production: signal_df = apply_signal_to_asset(df, config)
    signal_df = df.with_columns(
        signal_name=pl.lit("credit_vol").cast(pl.Categorical)
    )
    
    return signal_df


print("S5 Macro: VTRX Momentum (Credit Volatility)")
print("  - read_data_s5(): Load Bloomberg VTRX data (vtrxuhp)")
print("  - compute_score_s5(): close -> forward_fill -> pct_change -> ewm_mean")
print("  - apply_scaling(): z-score by EWM std -> clip to bounds")

S5 Macro: Credit Volatility - Spread volatility signal


## S6: VIX Momentum (Macro)

In [ ]:
"""
Macro Signal S6: VIX Momentum.

Score = EWM mean of VIX pct_change / EWM std (self-normalizing).
Data source: Bloomberg VIX data.
"""

def read_data_s6(config: dict) -> pl.DataFrame:
    """Load BBG VIX data (cached with full config tickers).
    
    In production: return read_vix_data(
        start_date=date.fromisoformat(config["start_date"]),
        end_date=date.fromisoformat(config["end_date"]),
        table=cfg["table"],
        tickers=cfg["tickers"],
    )
    """
    cfg = config["data"]["jade_tables"]["bbg_vix"]
    return config.get("vix_df", pl.DataFrame())


def compute_score_s6(vix_df: pl.DataFrame, lookback: int, vol_hl: int) -> pl.DataFrame:
    """VIX momentum score: EWM mean of pct_change / EWM std."""
    return (
        vix_df.filter(pl.col("asset") == "vix")
        .sort("date")
        .with_columns(
            pct_chg=(pl.col("close_price") / pl.col("close_price").shift(1) - 1.0)
        )
        .filter(pl.col("pct_chg").is_not_null())
        .with_columns(ewm=pl.col("pct_chg").ewm_mean(half_life=lookback))
        .with_columns(score=pl.col("ewm") / pl.col("ewm").ewm_std(half_life=vol_hl))
        .select("date", "score")
    )


def compute_signal_s6_vix_momentum(config: dict) -> pl.DataFrame:
    """S6 Macro: VIX Momentum
    
    Full production-style implementation.
    Self-normalizing score (ewm_mean / ewm_std), so apply_scaling only clips.
    """
    sig_cfg = config["macro_signals"]["vix_momentum"]
    clip_bounds = tuple(config["clip_bounds"])
    
    vix_df = read_data_s6(config)
    df = compute_score_s6(vix_df, lookback=sig_cfg["lookback"], vol_hl=sig_cfg["vol_hl"])
    df = apply_scaling(df, bounds=clip_bounds)  # clip only, already self-normalized
    
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("vix_momentum").cast(pl.Categorical)
    )
    
    return signal_df


print("S6 Macro: VIX Momentum")
print("  - read_data_s6(): Load Bloomberg VIX data")
print("  - compute_score_s6():")
print("    1. pct_chg = close_price / shift(1) - 1")
print("    2. ewm = pct_chg.ewm_mean(lookback)")
print("    3. score = ewm / ewm.ewm_std(vol_hl)")
print("  - apply_scaling(): clip only (already self-normalized)")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S6 Macro: VIX Momentum - Equity vol momentum (inverted)


## S7: MOVE Momentum (Macro)

In [ ]:
"""
Macro Signal S7: MOVE Momentum.

Score = EWM mean of MOVE pct_change, z-scored by EWM std.
Data source: Bloomberg MOVE index data.
"""

def read_data_s7(config: dict) -> pl.DataFrame:
    """Load BBG MOVE data (cached with full config tickers).
    
    In production: return read_bbg_data(
        start_date=date.fromisoformat(config["start_date"]),
        end_date=date.fromisoformat(config["end_date"]),
        table=cfg["table"],
        tickers=cfg["tickers"],
    )
    """
    cfg = config["data"]["jade_tables"]["bbg_move"]
    return config.get("move_df", pl.DataFrame())


def compute_score_s7(bbg_data: pl.DataFrame, lookback: int) -> pl.DataFrame:
    """MOVE momentum raw score: EWM mean of pct_change."""
    return (
        bbg_data.filter(pl.col("asset") == "move")
        .sort("date")
        .with_columns(
            score=(
                pl.col("close_price") / pl.col("close_price").shift(1) - 1.0
            ).ewm_mean(half_life=lookback)
        )
        .select("date", "score")
    )


def compute_signal_s7_move_momentum(config: dict) -> pl.DataFrame:
    """S7 Macro: MOVE Momentum
    
    Full production-style implementation.
    """
    sig_cfg = config["macro_signals"]["move_momentum"]
    clip_bounds = tuple(config["clip_bounds"])
    
    bbg_data = read_data_s7(config)
    df = compute_score_s7(bbg_data, lookback=sig_cfg["lookback"])
    df = apply_scaling(df, ewm_std_hl=sig_cfg["hl_std"], bounds=clip_bounds)
    
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("move_momentum").cast(pl.Categorical)
    )
    
    return signal_df


print("S7 Macro: MOVE Momentum")
print("  - read_data_s7(): Load Bloomberg MOVE index data")
print("  - compute_score_s7(): (close_price / shift(1) - 1) -> ewm_mean")
print("  - apply_scaling(): z-score by EWM std -> clip to bounds")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S7 Macro: MOVE Momentum - Rates vol momentum (inverted)


## S8: Spread Correlation (Macro)

In [ ]:
"""
Macro Signal S8: Spread-Correlation Value.

Kalman filter regression of CDXIG spread diff vs CORIM pct_change.
Score = negated Kalman residual, z-scored by EWM std.
Data source: JPM DQ (CDXIG spread) + BBG VIX (CORIM).
"""

def read_data_s8(config: dict) -> Tuple[pl.DataFrame, pl.DataFrame]:
    """Load CDXIG spread from JPM DQ and CORIM from BBG VIX (both cached).
    
    In production:
        jpm_df = read_jpm_dq_data(...)
        vix_df = read_vix_data(...)
    """
    # CDXIG spread
    cdxig_spread = (
        config.get("jpm_df", pl.DataFrame())
        .filter(pl.col("asset") == "cdxig")
        .select("date", "spread")
        .sort("date")
    )
    
    # CORIM data
    corim_data = (
        config.get("vix_df", pl.DataFrame())
        .filter(pl.col("asset") == "corim")
        .select("date", "close_price")
        .sort("date")
    )
    
    return cdxig_spread, corim_data


def compute_score_s8(
    cdxig_spread: pl.DataFrame,
    corim_data: pl.DataFrame,
    period: int,
) -> pl.DataFrame:
    """Kalman regression of spread diff vs CORIM pct_change. Score = negated residual."""
    # X = spread diff
    signal_x = (
        cdxig_spread.with_columns(x=pl.col("spread").diff())
        .select("date", "x")
        .drop_nulls()
    )
    
    # Y = CORIM pct_change
    signal_y = (
        corim_data.with_columns(y=pl.col("close_price").pct_change())
        .select("date", "y")
        .drop_nulls()
    )
    
    # Join and smooth with rolling mean
    df = (
        signal_x.join(signal_y, on="date", how="inner")
        .sort("date")
        .with_columns(
            pl.col("x").rolling_mean(window_size=period),
            pl.col("y").rolling_mean(window_size=period),
        )
        .drop_nulls()
    )
    
    # Kalman filter regression: y ~ beta * x + intercept
    state_means = compute_kalman_beta(
        df["x"].to_numpy(),
        df["y"].to_numpy(),
        delta=1.0e-5,
        use_constant=True,
    )
    
    # Extract slope and intercept from Kalman state
    df = df.with_columns(
        pl.Series("kslope", state_means[:, 0, 0]),
        pl.Series("kintercept", state_means[:, 0, 1]),
    ).with_columns(
        # Score = negated residual: -(y - predicted)
        score=-(pl.col("y") - pl.col("kslope") * pl.col("x") - pl.col("kintercept"))
    )
    
    return df.select("date", "score")


def compute_signal_s8_spread_correlation(config: dict) -> pl.DataFrame:
    """S8 Macro: Spread-Correlation Value
    
    Full production-style implementation using Kalman filter.
    """
    sig_cfg = config["macro_signals"]["spread_correlation"]
    clip_bounds = tuple(config["clip_bounds"])
    
    cdxig_spread, corim_data = read_data_s8(config)
    df = compute_score_s8(cdxig_spread, corim_data, period=sig_cfg["period"])
    df = apply_scaling(df, ewm_std_hl=sig_cfg["hl_std"], bounds=clip_bounds)
    
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("spread_correlation").cast(pl.Categorical)
    )
    
    return signal_df


print("S8 Macro: Spread-Correlation Value (Kalman)")
print("  - read_data_s8(): Load CDXIG spread (JPM DQ) + CORIM (BBG VIX)")
print("  - compute_score_s8():")
print("    1. x = spread.diff(), y = close_price.pct_change()")
print("    2. Rolling mean smoothing (window=period)")
print("    3. compute_kalman_beta(x, y, delta=1e-5, use_constant=True)")
print("    4. Extract kslope, kintercept from state_means")
print("    5. score = -(y - kslope*x - kintercept)")
print("  - apply_scaling(): z-score by EWM std -> clip to bounds")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S8 Macro: Spread Correlation - Credit-equity correlation regime


## S9: Cash-Synthetic Basis (Macro)

In [ ]:
"""
Macro Signal S9: Cash-Synthetic Basis.

Signal Logic: basis = CDX HY spread (ex-10) - HYG YAS spread.
Quantile filtered (suppress extreme positive basis), sign-based, inverted.
Data source: JPM DQ (CDX HY ex-10 spread) + BBG ETF (HYG YAS spread).
"""

def read_data_s9(config: dict) -> Tuple[pl.DataFrame, pl.DataFrame]:
    """Load CDX HY (ex-10) spread and HYG YAS spread.
    
    Returns:
        (syn_df, cash_df) - each with columns [date, value]
    """
    # Synthetic: CDX HY spread excluding top-10 widest names
    syn_df = (
        config.get("jpm_df", pl.DataFrame())
        .filter(pl.col("asset") == "cdxhy")
        .select("date", pl.col("spread_excl_10").alias("syn"))
        .sort("date")
    )
    
    # Cash: HYG yield-adjusted spread
    cash_df = (
        config.get("etf_df", pl.DataFrame())
        .filter(pl.col("asset") == "hyg")
        .select("date", pl.col("yas_yld_spread").alias("cash"))
        .sort("date")
    )
    
    return syn_df, cash_df


def compute_score_s9(
    syn_df: pl.DataFrame,
    cash_df: pl.DataFrame,
    period: int,
    q_level: float,
) -> pl.DataFrame:
    """Cash-synthetic basis score: -sign(basis), quantile filtered.
    
    basis = synthetic spread - cash spread
    Set basis to 0 where it exceeds rolling quantile (suppress extreme positive basis).
    Signal = -sign(basis).
    """
    df = syn_df.join(cash_df, on="date", how="inner").sort("date")
    df = df.with_columns(basis=pl.col("syn") - pl.col("cash"))
    
    # Quantile filter: suppress extreme positive basis (distress events)
    df = df.with_columns(
        basis_q=pl.col("basis").rolling_quantile(q_level, window_size=period)
    )
    df = df.with_columns(pl.col("basis_q").forward_fill())
    df = df.with_columns(
        basis=pl.when(pl.col("basis") > pl.col("basis_q"))
        .then(0.0)
        .otherwise(pl.col("basis"))
    )
    
    # Signal: negative of the sign of the basis
    df = df.with_columns(score=-1.0 * pl.col("basis").sign())
    df = df.select("date", "score")
    
    return df


def compute_signal_s9_cash_syn_basis(config: dict) -> pl.DataFrame:
    """S9 Macro: Cash-Synthetic Basis
    
    Full production-style implementation.
    No zscore - signal is already binary (-1/0/+1).
    """
    sig_cfg = config["macro_signals"]["cash_syn_basis"]
    
    syn_df, cash_df = read_data_s9(config)
    df = compute_score_s9(
        syn_df, cash_df, 
        period=sig_cfg["period"], 
        q_level=sig_cfg["q_level"]
    )
    # No zscore - signal is already binary (-1/0/+1)
    
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("cash_syn_basis").cast(pl.Categorical)
    )
    
    return signal_df


print("S9 Macro: Cash-Synthetic Basis")
print("  - read_data_s9(): Load CDX HY ex-10 spread (JPM DQ) + HYG YAS spread (ETF)")
print("  - compute_score_s9():")
print("    1. basis = syn - cash")
print("    2. basis_q = rolling_quantile(q_level, period)")
print("    3. Suppress: basis = 0 where basis > basis_q")
print("    4. score = -sign(basis)")
print("  - No zscore - signal is already binary (-1/0/+1)")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S9 Macro: Cash-Synthetic Basis - Cash vs CDS basis


## S10: Spread to Price (Macro)

In [ ]:
"""
Macro Signal S10: Spread-to-Price Value.

Kalman filter regression of HYG pct_change vs CDX HY spread pct_change.
Score = tanh(residual / EWM std smoothed by EWM mean + 1) / 2 (long-biased).
Data source: JPM DQ (CDX HY spread) + BBG ETF data (HYG close).
"""

def read_data_s10(config: dict) -> Tuple[pl.DataFrame, pl.DataFrame]:
    """Load CDX HY spread from JPM DQ and HYG close from ETF data (both cached).
    
    In production:
        jpm_df = read_jpm_dq_data(...)
        etf_df = read_etf_data(...)
    """
    # CDX HY spread
    cdx_spread = (
        config.get("jpm_df", pl.DataFrame())
        .filter(pl.col("asset") == "cdxhy")
        .select("date", "spread")
        .sort("date")
    )
    
    # HYG close
    hyg_close = (
        config.get("etf_df", pl.DataFrame())
        .filter(pl.col("asset") == "hyg")
        .select("date", "close")
        .sort("date")
    )
    
    return cdx_spread, hyg_close


def compute_score_s10(
    cdx_spread: pl.DataFrame,
    hyg_close: pl.DataFrame,
    hl_std: int,
    hl_smooth: int,
) -> pl.DataFrame:
    """Kalman regression of HYG pct_change vs spread pct_change. Long-biased tanh score."""
    # X = spread pct_change
    signal_x = (
        cdx_spread.with_columns(x=pl.col("spread").pct_change())
        .select("date", "x")
        .drop_nulls()
    )
    
    # Y = HYG pct_change
    signal_y = (
        hyg_close.with_columns(y=pl.col("close").pct_change())
        .select("date", "y")
        .drop_nulls()
    )
    
    # Join
    df = signal_x.join(signal_y, on="date", how="inner").sort("date")
    
    if df.height == 0:
        return pl.DataFrame({"date": [], "score": []}).cast(
            {"date": pl.Date, "score": pl.Float64}
        )
    
    # Kalman filter regression: y ~ beta * x + intercept
    state_means = compute_kalman_beta(
        df["x"].to_numpy(),
        df["y"].to_numpy(),
        delta=1.0e-3,
        use_constant=True,
    )
    
    # Extract slope and intercept
    df = df.with_columns(
        pl.Series("kslope", state_means[:, 0, 0]),
        pl.Series("kintercept", state_means[:, 0, 1]),
    )
    
    # Residual
    df = df.with_columns(
        residual=pl.col("y") - pl.col("kslope") * pl.col("x") - pl.col("kintercept")
    )
    
    # Z-score residual, then apply long-biased tanh transformation
    df = df.with_columns(
        score=(
            pl.col("residual")
            / pl.col("residual").ewm_std(half_life=hl_std, adjust=False)
        ).replace([float("inf"), float("-inf")], None)
    )
    
    # Smooth and apply tanh(score + 1) / 2 for long bias
    df = df.with_columns(
        score=(pl.col("score").ewm_mean(half_life=hl_smooth) + 1.0).tanh() / 2.0
    )
    
    return df.select("date", "score")


def compute_signal_s10_spread_to_price(config: dict) -> pl.DataFrame:
    """S10 Macro: Spread-to-Price Value
    
    Full production-style implementation using Kalman filter.
    No apply_scaling - tanh(score+1)/2 in compute_score replaces clip.
    """
    sig_cfg = config["macro_signals"]["spread_to_price"]
    
    cdx_spread, hyg_close = read_data_s10(config)
    df = compute_score_s10(
        cdx_spread, hyg_close, 
        hl_std=sig_cfg["hl_std"], 
        hl_smooth=sig_cfg["hl_smooth"]
    )
    # No apply_scaling - tanh(score+1)/2 in compute_score replaces clip
    
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("spread_to_price").cast(pl.Categorical)
    )
    
    return signal_df


print("S10 Macro: Spread-to-Price Value (Kalman)")
print("  - read_data_s10(): Load CDX HY spread (JPM DQ) + HYG close (ETF)")
print("  - compute_score_s10():")
print("    1. x = spread.pct_change(), y = close.pct_change()")
print("    2. compute_kalman_beta(x, y, delta=1e-3, use_constant=True)")
print("    3. residual = y - kslope*x - kintercept")
print("    4. z_score = residual / ewm_std(hl_std)")
print("    5. score = tanh(ewm_mean(z_score, hl_smooth) + 1) / 2 (long-biased)")
print("  - No apply_scaling - tanh replaces clip")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S10 Macro: Spread to Price - Credit vs equity relative value


## S11: VIX Term Structure (Macro)

In [ ]:
"""
Macro Signal S11: VIX Term Structure.

Signal Logic: UX6/UX1 - 1 slope, EWM smoothed, quantile filtered, sign inverted.
Already internally normalized - no external zscore.
Data source: Bloomberg futures data (VIX futures UX1-UX7).
"""

def read_data_s11(config: dict) -> pl.DataFrame:
    """Load futures data (cached with full config tickers), filter to VIX futures.
    
    In production: return read_futures_data(...)
    """
    vix_assets = ["ux1", "ux2", "ux3", "ux4", "ux5", "ux6", "ux7"]
    df = (
        config.get("futures_df", pl.DataFrame())
        .filter(pl.col("asset").is_in(vix_assets))
        .select("date", "asset", "close")
        .sort("date", "asset")
    )
    return df


def compute_score_s11(vix_df: pl.DataFrame, period: int, q_level: float) -> pl.DataFrame:
    """VIX term structure slope: UX6/UX1 - 1, EWM smoothed, quantile filtered, sign inverted."""
    df = (
        vix_df.pivot(index="date", on="asset", values="close")
        .sort("date")
        .with_columns(slope=(pl.col("ux6") / pl.col("ux1")) - 1)
        .with_columns(
            slope_q=pl.col("slope").rolling_quantile(q_level, window_size=period)
        )
        # Binary signal: +1 if slope >= quantile, -1 otherwise
        .with_columns(score=((pl.col("slope") >= pl.col("slope_q")).cast(pl.Float64) * 2.0) - 1.0)
    )
    df = df.select("date", "score")
    return df


def compute_signal_s11_vix_term_structure(config: dict) -> pl.DataFrame:
    """S11 Macro: VIX Term Structure
    
    Full production-style implementation.
    No zscore - signal is already internally normalized.
    """
    sig_cfg = config["macro_signals"]["vix_term_structure"]
    
    vix_df = read_data_s11(config)
    df = compute_score_s11(vix_df, period=sig_cfg["period"], q_level=sig_cfg["q_level"])
    # No zscore - signal is already internally normalized
    
    filter_cfg = config["filters"]
    de_filter = get_filter(
        date.fromisoformat(config["start_date"]),
        date.fromisoformat(config["end_date"]),
        config["data"],
        level=filter_cfg["level"],
        window=filter_cfg["window"],
        asset=filter_cfg["asset"],
    )
    
    df = apply_filter(df, de_filter)
    signal_df = apply_signal_to_asset(df, config)
    signal_df = signal_df.with_columns(
        signal_name=pl.lit("vix_term_structure").cast(pl.Categorical)
    )
    
    return signal_df


print("S11 Macro: VIX Term Structure")
print("  - read_data_s11(): Load VIX futures (ux1-ux7)")
print("  - compute_score_s11():")
print("    1. Pivot to wide format (date, ux1, ux2, ... ux7)")
print("    2. slope = ux6/ux1 - 1")
print("    3. slope_q = rolling_quantile(q_level, period)")
print("    4. score = (slope >= slope_q) * 2 - 1 (binary -1/+1)")
print("  - No zscore - signal is already internally normalized")
print("  - Full pipeline: get_filter -> apply_filter -> apply_signal_to_asset")

S11 Macro: VIX Term Structure - Vol term structure slope


---
# SINGLE-NAME SIGNALS (8)
Issuer/security-level signals derived from individual name data.
---

## S1: Spread Reversion (Micro)

In [15]:
def compute_signal_s1_spread_reversion_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S1 Micro: Spread Reversion
    
    Issuer-level spread mean reversion.
    Wide spreads relative to issuer history -> expect tightening.
    """
    return df.with_columns([
        (pl.col('issuer_spread') - pl.col('issuer_spread').ewm_mean(alpha=get_ewm_alpha(config.half_life_hl_mean))
         .over('issuer_id')).alias('_spread_dev')
    ]).pipe(
        compute_ewm_zscore, '_spread_dev', config, 's1_spread_reversion_micro'
    ).drop('_spread_dev').with_columns([
        (-pl.col('s1_spread_reversion_micro')).alias('s1_spread_reversion_micro')
    ])

print("S1 Micro: Spread Reversion - Issuer-level mean reversion")

S1 Micro: Spread Reversion - Issuer-level mean reversion


## S2: Equity Volatility (Micro)

In [16]:
def compute_signal_s2_equity_vol_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S2 Micro: Equity Volatility
    
    Issuer equity volatility.
    High equity vol -> credit risk, expect spread widening.
    """
    return df.with_columns([
        pl.col('issuer_equity_return').rolling_std(config.vol_lookback).over('issuer_id').alias('_eq_vol')
    ]).pipe(
        compute_ewm_zscore, '_eq_vol', config, 's2_equity_vol_micro'
    ).drop('_eq_vol').with_columns([
        # Negate: high vol -> short credit
        (-pl.col('s2_equity_vol_micro')).alias('s2_equity_vol_micro')
    ])

print("S2 Micro: Equity Volatility - Issuer equity vol signal")

S2 Micro: Equity Volatility - Issuer equity vol signal


## S3: DTD Momentum (Micro)

In [17]:
def compute_signal_s3_dtd_momentum_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S3 Micro: Distance-to-Default Momentum
    
    Momentum in Merton model distance-to-default.
    Rising DTD -> improving credit quality.
    """
    return df.with_columns([
        (pl.col('dtd') - pl.col('dtd').shift(config.lookback_days).over('issuer_id')).alias('_dtd_chg')
    ]).pipe(
        compute_ewm_zscore, '_dtd_chg', config, 's3_dtd_momentum_micro'
    ).drop('_dtd_chg')

print("S3 Micro: DTD Momentum - Distance-to-default momentum")

S3 Micro: DTD Momentum - Distance-to-default momentum


## S4: Spread Skew (Micro)

In [18]:
def compute_signal_s4_spread_skew_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S4 Micro: Spread Skew Comp/Decomp
    
    Skewness of spread changes.
    Negative skew -> tail risk, positive skew -> upside potential.
    """
    return df.with_columns([
        pl.col('issuer_spread').pct_change().rolling_skew(config.vol_lookback).over('issuer_id').alias('_skew')
    ]).pipe(
        compute_ewm_zscore, '_skew', config, 's4_spread_skew_micro'
    ).drop('_skew')

print("S4 Micro: Spread Skew - Spread change skewness")

S4 Micro: Spread Skew - Spread change skewness


## S5: Spread Dispersion (Micro)

In [19]:
def compute_signal_s5_spread_dispersion_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S5 Micro: Spread Dispersion
    
    Issuer spread relative to sector peers.
    Wide vs peers -> relative value opportunity.
    """
    return df.with_columns([
        # Spread vs sector median
        (pl.col('issuer_spread') - pl.col('issuer_spread').median().over('sector')).alias('_vs_sector')
    ]).pipe(
        compute_ewm_zscore, '_vs_sector', config, 's5_spread_dispersion_micro'
    ).drop('_vs_sector').with_columns([
        # Negate: wide vs peers -> expect convergence
        (-pl.col('s5_spread_dispersion_micro')).alias('s5_spread_dispersion_micro')
    ])

print("S5 Micro: Spread Dispersion - Relative value vs sector")

S5 Micro: Spread Dispersion - Relative value vs sector


## S6: Cash Index Basis (Micro)

In [20]:
def compute_signal_s6_cash_index_basis_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S6 Micro: Cash Index Basis
    
    Issuer cash spread vs CDX index spread.
    Wide basis -> single-name cheap vs index.
    """
    return df.with_columns([
        (pl.col('issuer_spread') - pl.col('cdx_spread')).alias('_basis')
    ]).pipe(
        compute_ewm_zscore, '_basis', config, 's6_cash_index_basis_micro'
    ).drop('_basis').with_columns([
        (-pl.col('s6_cash_index_basis_micro')).alias('s6_cash_index_basis_micro')
    ])

print("S6 Micro: Cash Index Basis - Single-name vs index basis")

S6 Micro: Cash Index Basis - Single-name vs index basis


## S7: Options Implied Volatility (Micro)

In [21]:
def compute_signal_s7_options_iv_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S7 Micro: Options Implied Volatility
    
    Issuer equity option implied vol.
    High IV -> market pricing risk, potential credit weakness.
    """
    return df.pipe(
        compute_ewm_zscore, 'issuer_option_iv', config, 's7_options_iv_micro'
    ).with_columns([
        # Negate: high IV -> short credit
        (-pl.col('s7_options_iv_micro')).alias('s7_options_iv_micro')
    ])

print("S7 Micro: Options IV - Equity option implied vol")

S7 Micro: Options IV - Equity option implied vol


## S8: Spread vs Equity (Micro)

In [22]:
def compute_signal_s8_spread_vs_equity_micro(df: pl.DataFrame, config: SignalConfig) -> pl.DataFrame:
    """S8 Micro: Spread vs Equity
    
    Relative move in spread vs equity.
    Spread widening without equity drop -> credit overreaction.
    """
    return df.with_columns([
        # Z-score of spread change
        pl.col('issuer_spread').pct_change().alias('_spread_ret'),
        # Z-score of equity change
        pl.col('issuer_equity_price').pct_change().alias('_eq_ret'),
    ]).with_columns([
        # Difference: spread move vs equity move
        (pl.col('_spread_ret') + pl.col('_eq_ret')).alias('_diff')  # Note: spread up is bad, equity up is good
    ]).pipe(
        compute_ewm_zscore, '_diff', config, 's8_spread_vs_equity_micro'
    ).drop(['_spread_ret', '_eq_ret', '_diff']).with_columns([
        # Negate: spread widening without equity drop -> expect spread tightening
        (-pl.col('s8_spread_vs_equity_micro')).alias('s8_spread_vs_equity_micro')
    ])

print("S8 Micro: Spread vs Equity - Credit-equity relative value")

S8 Micro: Spread vs Equity - Credit-equity relative value
